# 1.自定义层
深度学习成功背后的一个因素是神经网络的灵活性：我们可以用创造性的方式组合不同的层，从而设计出适用于各种任务的架构。

## 1.1 不带参数的层
首先，我们构造一个没有任何参数的自定义层。下面的CenteredLayer类要从其输入中减去均值。

In [1]:
import torch
import torch.nn.functional as F
from torch import nn

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [2]:
layer = CenteredLayer()
layer(torch.FloatTensor([1, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [3]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer()) # 我们可以将层作为组件合并到复杂的模型中
# CenteredLayer 可以自动处理输入的任意形状。

In [4]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(6.1700e-09, grad_fn=<MeanBackward0>)

## 1.2 带参数的层
在带参数的层中，这些参数可以通过训练进行调整。我们可以使用内置函数来创建参数，这些函数提供了一些基本的管理功能，比如管理访问、初始化、贡献、保存和加载模型参数。这样做的好处就是：**我们不需要为每个自定义层编写自定义的序列化程序。**

现在，我们实现自定义版本的全连接层。回想一下，该层需要两个参数，一个用于表示权重，另一个用于表示片中。在次实现中，我们使用修正线性单元作为激活函数。该层需要输入参数in_units和units，分别表示输入数和输出数。

In [6]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units, ))
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

In [7]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[ 2.1527,  0.2525,  0.1513],
        [ 1.4960,  2.2033,  1.3926],
        [ 0.0165, -0.5492, -1.2851],
        [-0.3151, -2.1970, -0.5732],
        [ 1.3207,  2.8999,  0.4343]], requires_grad=True)

In [9]:
linear(torch.rand(2, 5)) # 两个样本， 每个样本五个特征

# 实际执行顺序：
# 1. 调用 linear.__call__(torch.rand(2, 5))
# 2. __call__ 方法内部会做很多事情，然后调用 forward
# 3. forward 被执行


# 计算过程：
# (2, 5) @ (5, 3) = (2, 3)

tensor([[2.1399, 0.0000, 0.0000],
        [3.7996, 0.5109, 1.3006]])

In [10]:
torch.rand(2, 5)

tensor([[0.2609, 0.9677, 0.6208, 0.8265, 0.2510],
        [0.9965, 0.8445, 0.1545, 0.3411, 0.0574]])